<pre>
#+TITLE: Project 2: Analyzing the 2008 Financial Crisis
#+AUTHOR: Edgar Huamantla
#+DATE: 2026-03-22
</pre>

In [ ]:
# Because jupyter notebook has a global name space; use this area for imports
import pandas as pd
import numpy as np
import plotly.express as px
import math
from typing import Union

## SQLAlchemy imports
from sqlalchemy import create_engine
from sqlalchemy.types import Date, Float, String, Integer, Boolean

# aux packages
from IPython.display import Image
import logging

# set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Obtain resource files

In [ ]:
import pathlib
import importlib.util

#check if gdown is installed in current python environment; alert the user to activate their virtual environment to avoid installing to global
if importlib.util.find_spec("gdown") is None:
    print("gdown is not installed. Please activate your virtual environment and install gdown to proceed.")

# dictionary: filename and gdown id
file_names = {'USE3L0712.RSK': '1Y0WE4cqbZfdNEvFuWIDR_d6oDbHHNkly',
              'USE3L0812.RSK': '1Z3vdd5m8uoB4whomq92DQbZHJiKYJc0v'}

curr_working_dir = pathlib.Path.cwd()
print(f"Current working directory: {curr_working_dir}")

for file_name, gdown_id in file_names.items():
    file_path = curr_working_dir / file_name
    if not file_path.exists():
        print(f"File {file_path} does not exist.")
        !gdown {gdown_id} -O {file_path}
    else:
        print(f"File {file_path} already exists. No need to download.")




In [ ]:
# check file integrity of downloaded files
for file_count, file_name in enumerate(file_names.keys(), start=1):
    print(f"\nChecking integrity of file {file_count}: {file_name}...")
    ! head -n 3 {file_name}

In [ ]:
# extract, parse text files into pandas df. ignore first row with date (i.e. 31DEC2007)
barra_df_07 = pd.read_csv('USE3L0712.RSK', skiprows=1, header=0)
barra_df_08 = pd.read_csv('USE3L0812.RSK', skiprows=1, header=0)

In [ ]:
# check shape of each dataframe
print(f"{barra_df_07.shape=}")
print(f"{barra_df_08.shape=}")
# assert columns match
assert barra_df_07.columns.equals(barra_df_08.columns), "Columns of the two dataframes do not match."
pd.testing.assert_index_equal(barra_df_07.columns, barra_df_08.columns, check_names=True) # secondary way using pd testing module

In [ ]:
display(barra_df_07.head(3))
display(barra_df_08.head(3))

In [ ]:
# look for null counts
for df in [barra_df_07, barra_df_08]:
    print(f"{df.isnull().values.any()=}")

## Cleaning Dataframe:
- remove white spaces from:
    - column names
    - string values in columns
    - index values

In [ ]:
def clean_index_whitespace(df):
    match df.index:
        case pd.Index() as idx if idx.dtype == 'O': # object dtype, likely string index
            df.index = df.index.str.strip()
        case _:
            logging.info(f"Index {idx.dtype=}. Skipping")
    return df

def clean_col_whitespace(df):
    match df.columns.dtype:
        case dtype if dtype == 'O':
            # create the stripped version
            original_columns = df.columns
            stripped_columns = original_columns.str.strip()

            # find the difference to output
            changed_cols_mask = original_columns != stripped_columns
            cleaned_cols = original_columns[changed_cols_mask].tolist()

            if cleaned_cols:
                logging.info(f"Stripped whitespace from columns: {cleaned_cols}")
            else:
                logging.info("No columns had leading/trailing whitespace to clean.")

            df.columns = stripped_columns
        case _:
            logging.info(f"skipped: {df.columns.dtype=} is not object dtype")
    return df


In [ ]:
# clean df index for both dataframes
barra_df_07 = clean_index_whitespace(barra_df_07)
barra_df_08 = clean_index_whitespace(barra_df_08)

# clean df columns for both dataframes
barra_df_07 = clean_col_whitespace(barra_df_07)
barra_df_08 = clean_col_whitespace(barra_df_08)